In [1]:
!pip -q install pandas tqdm requests

In [ ]:
import pandas as pd
import requests
from tqdm.auto import tqdm

In [ ]:
API_KEY = "myAPI"
INPUT_CSV = ""
OUTPUT_CSV = ""

LANG_MAP = {
    "english": "en",
    "kyrgyz": "ky",
    "bengali": "bn",
    "swahili": "sw",
    "somali": "so",
    "luxembourgish": "lb",
    "irish": "ga",
}

def translate_v2_api(text, source_code=None, target="en"):
    if text is None or str(text).strip() == "":
        return ""
    url = "https://translation.googleapis.com/language/translate/v2"
    params = {"key": API_KEY}
    data = {"q": str(text), "target": target, "format": "text"}
    if source_code:
        data["source"] = source_code

    r = requests.post(url, params=params, data=data, timeout=60)
    if r.status_code != 200:
        return f"[TRANSLATION_ERROR] {r.status_code} {r.text[:300]}"
    return r.json()["data"]["translations"][0]["translatedText"]

df = pd.read_csv(INPUT_CSV)
if "response_translation" not in df.columns:
    df["response_translation"] = pd.NA

for i in tqdm(range(len(df)), desc="Translating"):
    cur = df.at[i, "response_translation"]
    if pd.notna(cur) and str(cur).strip() != "":
        continue

    lang_label = str(df.at[i, "language"]).strip().lower()
    text = df.at[i, "LLM_response"]

    if lang_label == "english":
        tr = str(text) if text is not None else ""
    else:
        src = LANG_MAP.get(lang_label)  # None => autodetect
        tr = translate_v2_api(text, source_code=src, target="en")

    df.at[i, "response_translation"] = tr

df.to_csv(OUTPUT_CSV, index=False)
print("Done:", OUTPUT_CSV)
